# Returns Data Cleaning

## Objective

Clean and validate product return transactions before loading into the analytics database.

## Cleaning Tasks

- Identify missing return information
- Validate return quantities
- Confirm invoice relationships
- Validate customer and product references
- Standardize return reasons
- Export cleaned returns data

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_path = Path("../data/raw/returns.csv")

clean_path = Path("../data/cleaned/returns_clean.csv")

In [3]:
returns = pd.read_csv(raw_path)

returns.head()

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason
0,1,533554,2026-01-11,1009,211.0,19,Damaged Product
1,2,509428,2025-01-30,1010,264.0,16,Quality Issue
2,3,500200,2025-10-05,1009,180.0,9,Expired Product
3,4,512448,2025-03-24,1013,174.0,8,Quality Issue
4,5,539490,2025-06-17,1001,148.0,2,Expired Product


In [4]:
returns.shape

(1000, 7)

In [5]:
returns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        1000 non-null   int64  
 1   invoice_id       1000 non-null   int64  
 2   return_date      1000 non-null   object 
 3   product_id       1000 non-null   int64  
 4   customer_id      1000 non-null   float64
 5   return_quantity  1000 non-null   int64  
 6   return_reason    999 non-null    object 
dtypes: float64(1), int64(4), object(2)
memory usage: 54.8+ KB


In [6]:
returns.isnull().sum()

return_id          0
invoice_id         0
return_date        0
product_id         0
customer_id        0
return_quantity    0
return_reason      1
dtype: int64

In [7]:
returns_clean = returns.copy()

In [8]:
returns_clean["return_id"].duplicated().sum()

0

In [9]:
returns_clean[
    returns_clean["return_quantity"] <= 0
]

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason
20,21,506114,2026-01-13,1002,140.0,-25,Customer Complaint


In [10]:
invoices_clean = pd.read_csv(
    "../data/cleaned/invoices_clean.csv"
)

In [11]:
invalid_invoices = returns_clean[
    ~returns_clean["invoice_id"]
    .isin(invoices_clean["invoice_id"])
]

invalid_invoices

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason
10,11,999999,2025-12-27,1024,493.0,4,Damaged Product


In [12]:
returns_clean[
    returns_clean["return_reason"].isna()
]

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason
30,31,536481,2025-10-20,1015,325.0,52,NaN


In [13]:
invoices_clean = pd.read_csv(
    "../data/cleaned/invoices_clean.csv"
)

In [14]:
invalid_invoices = returns_clean[
    ~returns_clean["invoice_id"]
    .isin(invoices_clean["invoice_id"])
]

invalid_invoices

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason
10,11,999999,2025-12-27,1024,493.0,4,Damaged Product


In [15]:
returns_clean["return_reason"] = (
    returns_clean["return_reason"]
    .fillna("Unknown")
)

In [16]:
returns_clean["return_quantity"] = (
    returns_clean["return_quantity"]
    .abs()
)

In [17]:
returns_clean.isnull().sum()

return_id          0
invoice_id         0
return_date        0
product_id         0
customer_id        0
return_quantity    0
return_reason      0
dtype: int64

In [18]:
returns_clean[
    returns_clean["return_quantity"] <= 0
]

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason


In [19]:
invalid_invoices = returns_clean[
    ~returns_clean["invoice_id"]
    .isin(invoices_clean["invoice_id"])
]

invalid_invoices

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason
10,11,999999,2025-12-27,1024,493.0,4,Damaged Product


In [20]:
returns_clean["invoice_exists"] = (
    returns_clean["invoice_id"]
    .isin(invoices_clean["invoice_id"])
)

In [21]:
returns_clean[
    returns_clean["invoice_exists"] == False
]

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason,invoice_exists
10,11,999999,2025-12-27,1024,493.0,4,Damaged Product,False


Returns Data Quality Findings:

- Identified 1 return referencing a nonexistent invoice.
- Preserved the record for investigation rather than deleting operational data.

In [22]:
products_clean = pd.read_csv(
    "../data/cleaned/products_clean.csv"
)

In [23]:
invalid_products = returns_clean[
    ~returns_clean["product_id"]
    .isin(products_clean["product_id"])
]

invalid_products

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason,invoice_exists


In [24]:
customers_clean = pd.read_csv(
    "../data/cleaned/customers_clean.csv"
)

In [25]:
invalid_customers = returns_clean[
    ~returns_clean["customer_id"]
    .isin(customers_clean["customer_id"])
]

invalid_customers

,return_id,invoice_id,return_date,product_id,customer_id,return_quantity,return_reason,invoice_exists


In [26]:
returns_clean = returns_clean.drop(
    columns=["invoice_exists"]
)

In [27]:
returns_clean.to_csv(
    clean_path,
    index=False
)

In [28]:
returns_clean = returns_clean.drop(
    columns=["invoice_exists"]
)

KeyError: "['invoice_exists'] not found in axis"

In [29]:
returns_clean.columns

Index(['return_id', 'invoice_id', 'return_date', 'product_id', 'customer_id',
       'return_quantity', 'return_reason'],
      dtype='object')

In [30]:
returns_clean.isnull().sum()

return_id          0
invoice_id         0
return_date        0
product_id         0
customer_id        0
return_quantity    0
return_reason      0
dtype: int64

In [31]:
(returns_clean["return_quantity"] <= 0).sum()

0

In [32]:
returns_clean["return_id"].duplicated().sum()

0

In [33]:
returns_clean.to_csv(
    clean_path,
    index=False
)

In [1]:
import pandas as pd

returns_clean = pd.read_csv(
    "../data/cleaned/returns_clean.csv"
)

In [2]:
returns_clean.dtypes

return_id            int64
invoice_id           int64
return_date         object
product_id           int64
customer_id        float64
return_quantity      int64
return_reason       object
dtype: object

In [3]:
returns_clean["customer_id"] = (
    returns_clean["customer_id"]
    .astype(int)
)

In [4]:
returns_clean.dtypes

return_id           int64
invoice_id          int64
return_date        object
product_id          int64
customer_id         int64
return_quantity     int64
return_reason      object
dtype: object

In [5]:
returns_clean.to_csv(
    "../data/cleaned/returns_clean.csv",
    index=False
)